#### Cell 1: Install Dependencies

In [2]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install transformers nltk pandas scikit-learn matplotlib seaborn sentencepiece
!pip install lettucedetect minicheck

Looking in indexes: https://download.pytorch.org/whl/cpu

ERROR: Could not find a version that satisfies the requirement minicheck (from versions: none)
ERROR: No matching distribution found for minicheck


#### Cell 2: Imports and Configuration

In [3]:
import torch
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score

nltk.download('punkt', quiet=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HHEM_THRESHOLD = 0.5

print(f"Using device: {DEVICE}")

Using device: cpu


#### Cell 3: Mock Dataset

In [4]:
mock_dataset = [
    {"id": "H01", "query": "Can I push/pull code via GitHub integration in Site24x7?", "retrieved_docs": ["Site24x7 GitHub integration supports monitoring repositories."], "response": "Yes, you can push and pull code using the GitHub integration in Site24x7.", "label": "hallucinated"},
    {"id": "H02", "query": "How do I set up alerts for GitHub commits in Site24x7?", "retrieved_docs": ["Site24x7 monitors GitHub for issues and PRs."], "response": "You can set up alerts for commits via webhook integration.", "label": "hallucinated"},
    # ... (keep your full 25 examples here)
    # Add all 13 hallucinated + 12 faithful examples
]

df_mock = pd.DataFrame(mock_dataset)
print(f"Mock dataset loaded: {len(df_mock)} examples")

Mock dataset loaded: 2 examples


#### Cell 4: Benchmark Dataset (RAGTruth subset)

In [5]:
ragtruth_dataset = [
    {"id": "B01", "query": "Sample query", "retrieved_docs": ["Relevant context"], "response": "Faithful answer", "label": "faithful"},
    # Add your 10 benchmark examples here
]

print(f"Benchmark dataset loaded: {len(ragtruth_dataset)} examples")

Benchmark dataset loaded: 1 examples


#### Cell 5: LettuceDetect Evaluation

In [7]:
from lettucedetect import HallucinationDetector

lettuce_results_mock = []
lettuce_results_ragtruth = []

print("Loading LettuceDetect...")
detector = HallucinationDetector(
    method="transformer",
    model_path="KRLabsOrg/lettucedetect-base-modernbert-en-v1"
)

print("Evaluating LettuceDetect on Mock Dataset:")
for row in mock_dataset:
    try:
        pred = detector.predict(
            context=row["retrieved_docs"], 
            question=row["query"], 
            answer=row["response"]
        )
        predicted = "hallucinated" if pred.get("is_hallucinated", False) else "faithful"
        lettuce_results_mock.append(predicted)
        ok = "Match" if predicted == row["label"] else "Mismatch"
        print(f"{ok} [{row['id']}] -> {predicted} | GT: {row['label']}")
    except Exception as e:
        print(f"Error on {row['id']}: {e}")
        lettuce_results_mock.append("error")

all_results["LettuceDetect"] = {"mock": lettuce_results_mock, "ragtruth": lettuce_results_ragtruth}
print("LettuceDetect evaluation completed.")

Loading LettuceDetect...


OSError: KRLabsOrg/lettucedetect-base-modernbert-en-v1 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

#### 